In [2]:
# ===============================
# Synthetic Dataset Generation
# ===============================

import numpy as np
import pandas as pd

np.random.seed(42)
N = 150

# -------------------------------
# Clinical Data
# -------------------------------
clinical = pd.DataFrame({
    "Age": np.random.normal(55, 12, N).clip(20, 85),
    "Sex": np.random.binomial(1, 0.4, N),  # 1=Male
    "Smoking": np.random.binomial(1, 0.35, N),
    "DiseaseDuration": np.random.exponential(5, N).clip(0.5, 20),
    "Comorbidities": np.random.poisson(2, N)
})

clinical["Autoantibody"] = np.random.choice(
    ["ANA", "Anti-Scl70", "Anti-Jo1", "Anti-RNP"], N
)
clinical["CTD_Type"] = np.random.choice(
    ["SSc", "RA", "PM/DM", "SLE"], N
)
clinical["Histology"] = np.random.choice(
    ["NSIP", "UIP", "OP"], N
)

# Generate FVC
base_fvc = (
    90
    - 0.6 * clinical["Age"]
    - 2.5 * clinical["Smoking"]
    - 1.5 * clinical["DiseaseDuration"]
    - 6 * (clinical["Histology"] == "UIP")
    - 5 * (clinical["Autoantibody"] == "Anti-Scl70")
)

clinical["FVC"] = (base_fvc + np.random.normal(0, 8, N)).clip(15, 95)

# Generate 6MWT
clinical["MWT"] = (
    600
    - 3 * clinical["Age"]
    - 30 * (clinical["FVC"] < 50)
    - 20 * (clinical["Histology"] == "UIP")
    + np.random.normal(0, 40, N)
).clip(100, 650)

# Severity labels
def fvc_class(x):
    if x >= 61:
        return 0
    elif x >= 31:
        return 1
    else:
        return 2

def mwt_class(x):
    if x < 250:
        return 2
    elif x <= 350:
        return 1
    else:
        return 0

clinical["FVC_Class"] = clinical["FVC"].apply(fvc_class)
clinical["MWT_Class"] = clinical["MWT"].apply(mwt_class)

clinical.to_csv("../data/clinical.csv", index=False)

# -------------------------------
# Radiology Data
# -------------------------------
radiology = pd.DataFrame({
    "GGO_Volume": np.random.gamma(2, 6, N),
    "Fibrosis_Volume": np.random.gamma(3, 5, N),
    "Honeycombing": np.random.binomial(1, 0.35, N),
    "Pulm_HTN": np.random.binomial(1, 0.25, N),
    "Segments_Involved": np.random.randint(1, 10, N)
})

def warrick_score(row):
    score = 0
    score += min(row["GGO_Volume"] / 10, 4)
    score += min(row["Fibrosis_Volume"] / 8, 5)
    score += 4 if row["Honeycombing"] else 0
    score += min(row["Segments_Involved"], 4)
    return int(score)

radiology["Warrick"] = radiology.apply(warrick_score, axis=1)

def warrick_class(x):
    if x <= 7:
        return 0
    elif x <= 15:
        return 1
    else:
        return 2

radiology["Warrick_Class"] = radiology["Warrick"].apply(warrick_class)
radiology.to_csv("../data/radiology.csv", index=False)

print("✔ Synthetic datasets generated")


✔ Synthetic datasets generated


ModuleNotFoundError: No module named 'sklearn'